# Stage 2.5 — Interactive full 2×2 decoder-arm reconstruction UI

Upload one AP and one lateral knee X-ray, select a completed fold, and compare all four controlled decoder arms using the **same fold-specific frozen front end**.

| Activation | Plain topology | Residual topology |
| --- | --- | --- |
| ReLU | U — `plain_unet_style` | `residual_relu_style` |
| PReLU | `plain_prelu_style` | V — `residual_vnet_style` |

The four viewers always show the complete predicted knee. CUDA is used when available; CPU fallback is supported for local review but 256-cubed four-arm inference can be slow. Selected bones are coloured; unselected bones remain as translucent grey context. The fixed decision threshold is 0.5, matching the training contract.

> **Research use only.** Models were trained on synthetic DRRs. Reconstructions from real clinical X-rays are qualitative, may be affected by domain shift, and must not be used for diagnosis. Uploaded images have no 3D ground truth, so this UI does not report Dice or surface metrics.


In [1]:
import atexit
import contextlib
import hashlib
import io
import json
import shutil
import tempfile
import time
import uuid
from pathlib import Path

import gradio as gr
import numpy as np
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import trimesh
from PIL import Image
from pydicom.pixels import apply_modality_lut, apply_voi_lut
from skimage import measure

def find_project_root(start):
    for candidate in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (candidate / 'configs' / 'baseline_protocol_v1.json').is_file():
            return candidate
        nested = candidate / 'TestProject'
        if (nested / 'configs' / 'baseline_protocol_v1.json').is_file():
            return nested
    raise FileNotFoundError('project root not found (expected configs/baseline_protocol_v1.json)')


# Local notebooks discover the project root from the current working directory.
ROOT = find_project_root(Path.cwd())
STAGE2_SCHEMA = 'foundation_stage2_v1'
DECODER_LOGIT_RESOLUTION = 128
TARGET_SIZE = 256
THRESHOLD = 0.5
SPACING_XYZ = (0.78125, 0.78125, 0.78125)
BONES = ['femur', 'tibia', 'patella', 'fibula']
FEATURE_CHANNELS = [64, 128, 256, 512]
PRETRAIN_MODEL = 'convnextv2_tiny.fcmae'
FUSION_TYPES = ['local', 'local', 'attention', 'attention']
SEED = 42

# Row-major display order: ReLU row, then PReLU row; plain column, then residual column.
ARM_FACTORS = {
    'plain_unet_style': {'activation': 'relu', 'residual': False, 'label': 'U — ReLU + plain'},
    'residual_relu_style': {'activation': 'relu', 'residual': True, 'label': 'ReLU + residual'},
    'plain_prelu_style': {'activation': 'prelu', 'residual': False, 'label': 'PReLU + plain'},
    'residual_vnet_style': {'activation': 'prelu', 'residual': True, 'label': 'V — PReLU + residual'},
}
DISPLAY_ARMS = list(ARM_FACTORS)
BONE_RGBA = {
    'femur': [76, 120, 168, 245],
    'tibia': [245, 133, 24, 245],
    'patella': [84, 162, 75, 245],
    'fibula': [228, 87, 86, 245],
}
MUTED_RGBA = [170, 170, 170, 55]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if BONES != ['femur', 'tibia', 'patella', 'fibula']:
    raise RuntimeError('canonical bone-channel order changed')


def clear_device_cache():
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def reset_peak_memory_stats():
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()


def synchronize_device():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def peak_device_memory_bytes():
    return int(torch.cuda.max_memory_allocated()) if DEVICE.type == 'cuda' else None


device_name = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU fallback'
print({'root': str(ROOT), 'device': device_name, 'gradio': gr.__version__, 'trimesh': trimesh.__version__})
if DEVICE.type == 'cpu':
    print('WARNING: CUDA is unavailable; full four-arm 256-cubed inference may be very slow on CPU.')


{'root': 'C:\\Users\\Chan Zheng Shao\\OneDrive\\Desktop\\Github Repo\\TestProject\\TestProject', 'device': 'CPU fallback', 'gradio': '6.17.3', 'trimesh': '4.12.2'}


## Inference architecture contract

These definitions are the inference-only subset of `03b_decoder_cross_validation.ipynb`. Strict state loading and the preflight tests below detect drift from the training notebook.


In [2]:
def make_activation(kind, channels):
    if kind == 'relu':
        return nn.ReLU(inplace=True)
    if kind == 'prelu':
        return nn.PReLU(channels)
    raise ValueError(f'unknown activation {kind!r}')


class DecoderBlock(nn.Module):
    def __init__(self, input_channels, output_channels, activation, residual):
        super().__init__()
        self.residual = residual
        self.proj = ((nn.Conv3d(input_channels, output_channels, 1)
                      if input_channels != output_channels else nn.Identity())
                     if residual else None)
        self.conv1 = nn.Conv3d(input_channels, output_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, output_channels)
        self.act1 = make_activation(activation, output_channels)
        self.conv2 = nn.Conv3d(output_channels, output_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, output_channels)
        self.act2 = make_activation(activation, output_channels)

    def forward(self, value):
        residual = self.proj(value) if self.residual else None
        value = self.act1(self.norm1(self.conv1(value)))
        value = self.norm2(self.conv2(value))
        if self.residual:
            value = value + residual
        return self.act2(value)


def block_for(arm, input_channels, output_channels):
    if arm not in ARM_FACTORS:
        raise ValueError(f'unknown arm {arm!r}')
    factors = ARM_FACTORS[arm]
    return DecoderBlock(input_channels, output_channels, factors['activation'], factors['residual'])


class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, query_map, context_map):
        batch, channels, height, width = query_map.shape
        query = query_map.flatten(2).transpose(1, 2)
        context = context_map.flatten(2).transpose(1, 2)
        attention = torch.softmax(
            self.query(query) @ self.key(context).transpose(-2, -1) * self.scale,
            dim=-1,
        )
        return (attention @ self.value(context) + query).transpose(1, 2).reshape(
            batch, channels, height, width
        )


class LocalFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, 3, padding=1)

    def forward(self, query_map, context_map):
        return self.mix(torch.cat([query_map, context_map], dim=1)) + query_map


class BiPlanarFrontEnd(nn.Module):
    def __init__(self, encoder_state, pretrained_configuration):
        super().__init__()
        self.encoder = timm.create_model(PRETRAIN_MODEL, pretrained=False, features_only=True)
        self.encoder.load_state_dict(encoder_state, strict=True)
        self.pretrained_configuration = pretrained_configuration
        channels = self.encoder.feature_info.channels()
        self.fusion = nn.ModuleList([
            CrossAttention(channel) if kind == 'attention' else LocalFusion(channel)
            for channel, kind in zip(channels, FUSION_TYPES)
        ])
        self.project_2d = nn.ModuleList([
            nn.Conv2d(source, target, 1)
            for source, target in zip(channels, FEATURE_CHANNELS)
        ])
        self.fuse_3d = nn.ModuleList([
            nn.Conv3d(2 * channel, channel, 3, padding=1)
            for channel in FEATURE_CHANNELS
        ])

    def normalize(self, raw):
        image = raw.repeat(1, 3, 1, 1)
        mean = torch.as_tensor(
            self.pretrained_configuration['mean'], device=image.device, dtype=image.dtype
        ).view(1, 3, 1, 1)
        std = torch.as_tensor(
            self.pretrained_configuration['std'], device=image.device, dtype=image.dtype
        ).view(1, 3, 1, 1)
        return (image - mean) / std

    @staticmethod
    def lift(ap_feature, lat_feature, projection, fusion3d):
        ap = projection(ap_feature)
        lat = projection(lat_feature).flip(3)
        batch, channels, size, _ = ap.shape
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(batch, channels, size, size, size)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(batch, channels, size, size, size)
        return fusion3d(torch.cat([ap_cube, lat_cube], dim=1))

    def forward(self, ap_raw, lat_raw):
        ap_levels = self.encoder(self.normalize(ap_raw))
        lat_levels = self.encoder(self.normalize(lat_raw))
        output = []
        for ap, lat, fusion, project, fuse3d in zip(
            ap_levels, lat_levels, self.fusion, self.project_2d, self.fuse_3d
        ):
            output.append(self.lift(fusion(ap, lat), fusion(lat, ap), project, fuse3d))
        return output


class MatchedDecoder3D(nn.Module):
    def __init__(self, arm):
        super().__init__()
        self.arm = arm
        c0, c1, c2, c3 = FEATURE_CHANNELS
        self.up3 = nn.ConvTranspose3d(c3, c2, 2, 2)
        self.dec3 = block_for(arm, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, 2, 2)
        self.dec2 = block_for(arm, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, 2, 2)
        self.dec1 = block_for(arm, c0 + c0, c0)
        self.refine128 = block_for(arm, c0, 32)
        self.output128 = nn.Conv3d(32, len(BONES), 1)

    def forward(self, features):
        level0, level1, level2, level3 = features
        value = self.dec3(torch.cat([self.up3(level3), level2], dim=1))
        value = self.dec2(torch.cat([self.up2(value), level1], dim=1))
        value = self.dec1(torch.cat([self.up1(value), level0], dim=1))
        value = self.refine128(F.interpolate(
            value,
            size=(DECODER_LOGIT_RESOLUTION,) * 3,
            mode='trilinear',
            align_corners=False,
        ))
        logits = self.output128(value)
        return F.interpolate(
            logits, size=(TARGET_SIZE,) * 3, mode='trilinear', align_corners=False
        )


def amp_context():
    if DEVICE.type != 'cuda':
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast('cuda', dtype=dtype)


## Artifact discovery and strict loading

Only complete four-arm folds are offered. Missing artifacts make a fold unavailable; present-but-invalid artifacts are reported as integrity failures and never silently skipped.


For local execution, the resolver accepts both `fold_0` (current HPC layout) and `fold0` (local artifact layout). It prefers `fcmae_p1_config.json` and falls back to the verified `pretrained_configuration` in `fcmae_p1_provenance.json`. Missing external shared-front-end provenance is replaced by strict embedded checkpoint identity, P2 file-hash, P2 state-hash, and decoder-recorded shared-file-hash validation.

In [3]:
_FILE_HASH_CACHE = {}
_FRONTEND_CACHE = {'fold': None, 'model': None, 'shared_sha256': None}


def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in _FILE_HASH_CACHE:
        digest = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b''):
                digest.update(chunk)
        _FILE_HASH_CACHE[key] = digest.hexdigest()
    return _FILE_HASH_CACHE[key]


def canonical_sha256(payload):
    encoded = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(encoded).hexdigest()


def tensor_sha256(tensor):
    value = tensor.detach().cpu().contiguous()
    digest = hashlib.sha256()
    digest.update(str(value.dtype).encode())
    digest.update(np.asarray(value.shape, dtype=np.int64).tobytes())
    digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def stage2_state_sha256(state):
    digest = hashlib.sha256()
    for name in sorted(state):
        digest.update(name.encode())
        digest.update(tensor_sha256(state[name]).encode())
    return digest.hexdigest()


def decoder_run_paths(fold, arm):
    run_dir = ROOT / 'models' / 'decoders' / STAGE2_SCHEMA / f'fold_{fold}' / arm
    return {
        'config': run_dir / 'config.json',
        'summary': run_dir / 'run_summary.json',
        'checkpoint': run_dir / 'best_decoder.pth',
    }


def frontend_artifact_paths(fold):
    base = ROOT / 'models' / STAGE2_SCHEMA
    preferred = base / f'fold_{fold}'
    local_legacy = base / f'fold{fold}'
    fold_root = preferred if preferred.is_dir() else local_legacy if local_legacy.is_dir() else preferred
    config_candidates = [
        fold_root / 'fcmae_p1_config.json',
        fold_root / 'fcmae_p1_provenance.json',
    ]
    config_path = next((path for path in config_candidates if path.is_file()), config_candidates[0])
    return {
        'root': fold_root,
        'layout': 'fold_with_underscore' if fold_root == preferred else 'local_fold_without_underscore',
        'p2': fold_root / 'fcmae_p2_encoder.pth',
        'config': config_path,
        'shared': fold_root / 'shared_frontend.pth',
        'provenance': fold_root / 'shared_frontend_provenance.json',
    }


def validate_run_artifacts(fold, arm):
    paths = decoder_run_paths(fold, arm)
    missing = [name for name, path in paths.items() if not path.is_file()]
    if missing:
        return None, f'missing {", ".join(missing)}'

    config = json.loads(paths['config'].read_text(encoding='utf-8'))
    summary = json.loads(paths['summary'].read_text(encoding='utf-8'))
    factors = ARM_FACTORS[arm]
    expected = {
        'schema_version': STAGE2_SCHEMA,
        'stage': 'controlled_decoder_cv',
        'fold': fold,
        'arm': arm,
        'activation': factors['activation'],
        'residual': factors['residual'],
        'seed': SEED,
        'sweep': False,
        'logit_resolution': DECODER_LOGIT_RESOLUTION,
        'output_resolution': TARGET_SIZE,
    }
    for key, value in expected.items():
        if config.get(key) != value or summary.get(key) != value:
            raise RuntimeError(f'protocol mismatch fold={fold} arm={arm} field={key}')
    if summary.get('success') is not True:
        raise RuntimeError(f'run is not successful: fold={fold} arm={arm}')
    config_sha = canonical_sha256(config)
    checkpoint_sha = sha256_file(paths['checkpoint'])
    if summary.get('config_sha256') != config_sha:
        raise RuntimeError(f'config hash mismatch: fold={fold} arm={arm}')
    if summary.get('checkpoint_sha256') != checkpoint_sha:
        raise RuntimeError(f'checkpoint hash mismatch: fold={fold} arm={arm}')
    if summary.get('hyperparameters', {}).get('threshold') != THRESHOLD:
        raise RuntimeError(f'threshold contract mismatch: fold={fold} arm={arm}')

    frontend_paths = frontend_artifact_paths(fold)
    required_frontend = {
        name: frontend_paths[name]
        for name in ('p2', 'config', 'shared')
        if not frontend_paths[name].is_file()
    }
    if required_frontend:
        missing_names = ', '.join(f'{name}={path.name}' for name, path in required_frontend.items())
        return None, f'missing front-end artifact(s): {missing_names}'
    configuration_payload = json.loads(frontend_paths['config'].read_text(encoding='utf-8'))
    if not isinstance(configuration_payload.get('pretrained_configuration'), dict):
        raise RuntimeError(f'front-end configuration has no pretrained_configuration: fold={fold}')
    shared_sha = sha256_file(frontend_paths['shared'])
    if config.get('shared_frontend_sha256') != shared_sha:
        raise RuntimeError(f'shared-front-end hash mismatch: fold={fold} arm={arm}')
    return {
        'fold': fold,
        'arm': arm,
        'config_sha256': config_sha,
        'checkpoint_sha256': checkpoint_sha,
        'shared_frontend_sha256': shared_sha,
        'frontend_layout': frontend_paths['layout'],
        'frontend_paths': frontend_paths,
        'paths': paths,
    }, None


def discover_complete_folds():
    complete = []
    diagnostics = {}
    identities = {}
    for fold in range(5):
        fold_identities = {}
        problems = []
        for arm in DISPLAY_ARMS:
            try:
                identity, problem = validate_run_artifacts(fold, arm)
            except Exception as exc:
                identity, problem = None, f'integrity failure: {type(exc).__name__}: {exc}'
            if identity is None:
                problems.append(f'{arm}: {problem}')
            else:
                fold_identities[arm] = identity
        if len(fold_identities) == len(DISPLAY_ARMS):
            shared_hashes = {item['shared_frontend_sha256'] for item in fold_identities.values()}
            if len(shared_hashes) != 1:
                problems.append('four arms do not share one front-end hash')
            else:
                complete.append(fold)
                identities[fold] = fold_identities
        diagnostics[fold] = problems or ['PASS — complete four-arm fold']
    return complete, diagnostics, identities


def validate_checkpoint_payload(payload, identity):
    expected = {
        'schema_version': STAGE2_SCHEMA,
        'stage': 'controlled_decoder_cv',
        'fold': identity['fold'],
        'arm': identity['arm'],
        'seed': SEED,
        'config_sha256': identity['config_sha256'],
        'shared_frontend_sha256': identity['shared_frontend_sha256'],
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'checkpoint payload mismatch for {key}: {identity["arm"]}')
    if not isinstance(payload.get('decoder_state'), dict):
        raise RuntimeError(f'checkpoint has no decoder_state: {identity["arm"]}')


def load_decoder(identity, device='cpu'):
    payload = torch.load(identity['paths']['checkpoint'], map_location='cpu', weights_only=False)
    validate_checkpoint_payload(payload, identity)
    decoder = MatchedDecoder3D(identity['arm'])
    decoder.load_state_dict(payload['decoder_state'], strict=True)
    if decoder.output128.out_channels != len(BONES):
        raise RuntimeError(f'four-channel head contract failed: {identity["arm"]}')
    return decoder.to(device).eval()


def load_frontend(fold, shared_sha):
    if _FRONTEND_CACHE['fold'] == fold and _FRONTEND_CACHE['shared_sha256'] == shared_sha:
        return _FRONTEND_CACHE['model']

    old_model = _FRONTEND_CACHE.get('model')
    if old_model is not None:
        old_model.to('cpu')
        del old_model
        clear_device_cache()

    paths = frontend_artifact_paths(fold)
    for name in ('p2', 'config', 'shared'):
        if not paths[name].is_file():
            raise FileNotFoundError(paths[name])
    if sha256_file(paths['shared']) != shared_sha:
        raise RuntimeError(f'front-end changed after fold discovery: fold={fold}')

    p2 = torch.load(paths['p2'], map_location='cpu', weights_only=False)
    shared = torch.load(paths['shared'], map_location='cpu', weights_only=False)
    configuration_payload = json.loads(paths['config'].read_text(encoding='utf-8'))
    configuration = configuration_payload.get('pretrained_configuration')
    if not isinstance(configuration, dict):
        raise RuntimeError(f'pretrained configuration is missing: {paths["config"]}')
    if not isinstance(p2.get('encoder_state'), dict):
        raise RuntimeError(f'P2 checkpoint has no encoder_state: fold={fold}')
    if shared.get('schema_version') != STAGE2_SCHEMA or shared.get('fold') != fold or shared.get('stage') != 'shared_frontend':
        raise RuntimeError(f'invalid shared-front-end payload: fold={fold}')
    if shared.get('p2_encoder_sha256') != sha256_file(paths['p2']):
        raise RuntimeError(f'P2 encoder file hash mismatch: fold={fold}')
    provenance_mode = 'embedded_checkpoint'
    if paths['provenance'].is_file():
        provenance = json.loads(paths['provenance'].read_text(encoding='utf-8'))
        if provenance.get('fold') != fold or provenance.get('checkpoint_sha256') != shared_sha:
            raise RuntimeError(f'shared-front-end provenance mismatch: fold={fold}')
        provenance_mode = 'external_json_and_embedded_checkpoint'

    model = BiPlanarFrontEnd(p2['encoder_state'], configuration)
    if shared.get('p2_encoder_state_sha256') != stage2_state_sha256(model.encoder.state_dict()):
        raise RuntimeError(f'instantiated P2 encoder state hash mismatch: fold={fold}')
    model.load_state_dict(shared['front_end_state'], strict=True)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(DEVICE).eval()
    _FRONTEND_CACHE.update(fold=fold, model=model, shared_sha256=shared_sha)
    print({'fold': fold, 'front_end_layout': paths['layout'], 'provenance_validation': provenance_mode})
    return model


## Privacy-preserving upload preprocessing

Uploads enter callbacks as bytes rather than paths. DICOM identity fields and original filenames are never returned or logged. Raw images preserve aspect ratio and are zero-padded to 256×256; exact NPY inputs must already satisfy the model contract.


In [4]:
def robust_normalize(array, lower=1.0, upper=99.0):
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 3:
        if array.shape[-1] in (3, 4):
            array = array[..., :3].mean(axis=-1)
        elif array.shape[0] == 1:
            array = array[0]
        else:
            raise ValueError(f'expected one 2D image, received shape {array.shape}')
    if array.ndim != 2 or min(array.shape) < 2:
        raise ValueError(f'expected a 2D image, received shape {array.shape}')
    if not np.isfinite(array).all():
        raise ValueError('image contains NaN or infinite values')
    low, high = np.percentile(array, [lower, upper])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        raise ValueError('image has no usable intensity range')
    return np.clip((array - low) / (high - low), 0.0, 1.0).astype(np.float32)


def resize_and_pad(array, size=TARGET_SIZE):
    array = np.asarray(array, dtype=np.float32)
    height, width = array.shape
    scale = min(size / height, size / width)
    new_height = max(1, min(size, int(round(height * scale))))
    new_width = max(1, min(size, int(round(width * scale))))
    tensor = torch.from_numpy(array)[None, None]
    resized = F.interpolate(
        tensor, size=(new_height, new_width), mode='bilinear', align_corners=False
    )[0, 0].numpy()
    output = np.zeros((size, size), dtype=np.float32)
    top = (size - new_height) // 2
    left = (size - new_width) // 2
    output[top:top + new_height, left:left + new_width] = resized
    return output


def dicom_view_warning(dataset, expected_view):
    fields = ('ViewPosition', 'SeriesDescription', 'ProtocolName')
    token = ' '.join(str(getattr(dataset, field, '')) for field in fields).upper()
    has_ap = 'AP' in token
    has_lat = 'LAT' in token or 'LATERAL' in token
    if expected_view == 'AP' and has_lat and not has_ap:
        return 'DICOM header appears lateral but was uploaded in the AP slot.'
    if expected_view == 'LAT' and has_ap and not has_lat:
        return 'DICOM header appears AP but was uploaded in the lateral slot.'
    if not has_ap and not has_lat:
        return f'DICOM header does not identify the {expected_view} view; slot assignment was trusted.'
    return None


def dicom_pixels(dataset):
    pixels = apply_modality_lut(dataset.pixel_array, dataset)
    used_voi = False
    if hasattr(dataset, 'WindowCenter') and hasattr(dataset, 'WindowWidth'):
        try:
            pixels = apply_voi_lut(pixels, dataset)
            used_voi = True
        except Exception:
            used_voi = False
    pixels = np.asarray(pixels, dtype=np.float32)
    if pixels.ndim == 3 and pixels.shape[0] == 1:
        pixels = pixels[0]
    if str(getattr(dataset, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        pixels = pixels.max() + pixels.min() - pixels
    return pixels, used_voi


def load_exact_npy(blob):
    array = np.load(io.BytesIO(blob), allow_pickle=False)
    if array.shape != (TARGET_SIZE, TARGET_SIZE):
        raise ValueError(f'NPY must have shape 256×256, received {array.shape}')
    if not np.issubdtype(array.dtype, np.number) or not np.isfinite(array).all():
        raise ValueError('NPY must contain only finite numeric values')
    array = array.astype(np.float32)
    if float(array.min()) < 0.0 or float(array.max()) > 1.0:
        raise ValueError('NPY values must lie in [0,1]')
    if float(array.max()) <= float(array.min()):
        raise ValueError('NPY image is constant')
    return array


def preprocess_upload(blob, expected_view):
    if blob is None:
        raise ValueError(f'upload the {expected_view} image')
    if not isinstance(blob, (bytes, bytearray)):
        raise TypeError('upload callback expected file bytes')
    raw = bytes(blob)
    warnings = []

    if raw.startswith(b'\x93NUMPY'):
        return load_exact_npy(raw), warnings, 'exact NPY'

    try:
        dataset = pydicom.dcmread(io.BytesIO(raw), force=False)
        if not hasattr(dataset, 'PixelData'):
            raise ValueError('DICOM has no PixelData')
        pixels, used_voi = dicom_pixels(dataset)
        warning = dicom_view_warning(dataset, expected_view)
        if warning:
            warnings.append(warning)
        normalized = robust_normalize(pixels)
        return resize_and_pad(normalized), warnings, ('DICOM VOI' if used_voi else 'DICOM percentile')
    except pydicom.errors.InvalidDicomError:
        pass

    try:
        with Image.open(io.BytesIO(raw)) as image:
            pixels = np.asarray(image)
    except Exception as exc:
        raise ValueError('unsupported or unreadable upload; use DICOM, PNG, JPEG, TIFF, or NPY') from exc
    return resize_and_pad(robust_normalize(pixels)), warnings, 'image percentile'


## Mesh construction and session isolation

Each prediction becomes four cached meshes. Highlight changes only rebuild lightweight GLB scenes; model inference is not repeated. Session cleanup removes derived meshes and generated GLBs.


In [5]:
UI_TEMP_ROOT = Path(tempfile.mkdtemp(prefix='knee_2x2_ui_'))
SESSION_CACHE = {}
atexit.register(shutil.rmtree, UI_TEMP_ROOT, ignore_errors=True)


def new_session():
    session_id = uuid.uuid4().hex
    session_dir = UI_TEMP_ROOT / session_id
    session_dir.mkdir(parents=True, exist_ok=False)
    SESSION_CACHE[session_id] = {'dir': session_dir, 'meshes': None, 'glbs': []}
    return session_id


def cleanup_session(session_id):
    record = SESSION_CACHE.pop(session_id, None)
    if record is not None:
        shutil.rmtree(record['dir'], ignore_errors=True)


def volume_to_mesh(probability, threshold=THRESHOLD):
    probability = np.asarray(probability, dtype=np.float32)
    if probability.shape != (TARGET_SIZE,) * 3:
        raise ValueError(f'probability volume has wrong shape: {probability.shape}')
    below = float(probability.min()) < threshold
    above = float(probability.max()) > threshold
    if not (below and above) or int((probability > threshold).sum()) < 10:
        return None
    try:
        vertices, faces, _, _ = measure.marching_cubes(
            probability,
            level=threshold,
            spacing=SPACING_XYZ,
            step_size=2,
            allow_degenerate=False,
        )
    except (RuntimeError, ValueError):
        return None
    center = (np.asarray(probability.shape, dtype=np.float32) - 1.0) * np.asarray(SPACING_XYZ) / 2.0
    return vertices.astype(np.float32) - center, faces.astype(np.int64)


def build_scene(arm_meshes, highlighted):
    selected = set(highlighted or [])
    scene = trimesh.Scene()
    for bone in BONES:
        geometry = arm_meshes.get(bone)
        if geometry is None:
            continue
        vertices, faces = geometry
        rgba = BONE_RGBA[bone] if bone in selected else MUTED_RGBA
        material = trimesh.visual.material.PBRMaterial(
            name=f'{bone}_material',
            baseColorFactor=np.asarray(rgba, dtype=np.uint8),
            metallicFactor=0.0,
            roughnessFactor=0.85,
            alphaMode='BLEND',
            doubleSided=True,
        )
        mesh = trimesh.Trimesh(vertices=vertices.copy(), faces=faces.copy(), process=False)
        mesh.visual = trimesh.visual.TextureVisuals(material=material)
        scene.add_geometry(mesh, node_name=bone, geom_name=bone)
    return scene


def render_scenes(session_id, highlighted):
    record = SESSION_CACHE.get(session_id)
    if record is None or record.get('meshes') is None:
        return (None,) * len(DISPLAY_ARMS)
    for old_path in record.get('glbs', []):
        old_path = Path(old_path)
        if old_path.is_file() and old_path.parent == record['dir']:
            old_path.unlink()
    outputs = []
    token = uuid.uuid4().hex[:8]
    for arm in DISPLAY_ARMS:
        scene = build_scene(record['meshes'][arm], highlighted)
        if not scene.geometry:
            outputs.append(None)
            continue
        output_path = record['dir'] / f'{arm}_{token}.glb'
        scene.export(file_obj=output_path, file_type='glb')
        outputs.append(str(output_path))
    record['glbs'] = [path for path in outputs if path is not None]
    return tuple(outputs)


## CPU-safe contract tests

These tests validate the 2×2 mapping, preprocessing edge cases, architecture shape contract, mesh materials, GLB export, and empty-channel behavior without running a 256³ forward pass.


In [6]:
def factorial_contract_test():
    expected = [
        ('plain_unet_style', 'relu', False),
        ('residual_relu_style', 'relu', True),
        ('plain_prelu_style', 'prelu', False),
        ('residual_vnet_style', 'prelu', True),
    ]
    observed = [
        (arm, ARM_FACTORS[arm]['activation'], ARM_FACTORS[arm]['residual'])
        for arm in DISPLAY_ARMS
    ]
    assert observed == expected
    for arm in DISPLAY_ARMS:
        model = MatchedDecoder3D(arm)
        assert model.output128.out_channels == 4
        assert len([layer for layer in model.modules() if isinstance(layer, nn.GroupNorm)]) == 8
    return {'layout': observed, 'channels': BONES}


def preprocessing_contract_test():
    exact = np.linspace(0, 1, TARGET_SIZE * TARGET_SIZE, dtype=np.float32).reshape(TARGET_SIZE, TARGET_SIZE)
    buffer = io.BytesIO()
    np.save(buffer, exact)
    loaded, warnings, source = preprocess_upload(buffer.getvalue(), 'AP')
    assert np.array_equal(loaded, exact) and warnings == [] and source == 'exact NPY'

    portrait = robust_normalize(np.arange(80 * 40, dtype=np.float32).reshape(80, 40))
    padded = resize_and_pad(portrait)
    assert padded.shape == (256, 256) and np.allclose(padded[:, :64], 0) and np.allclose(padded[:, -64:], 0)

    for invalid in (np.zeros((8, 8), dtype=np.float32), np.full((8, 8), np.nan, dtype=np.float32)):
        try:
            robust_normalize(invalid)
            raise AssertionError('invalid image was accepted')
        except ValueError:
            pass
    bad = io.BytesIO()
    np.save(bad, np.zeros((32, 32), dtype=np.float32))
    try:
        preprocess_upload(bad.getvalue(), 'LAT')
        raise AssertionError('wrong-shape NPY was accepted')
    except ValueError:
        pass

    raw = np.asarray([[0.0, 1.0], [2.0, 3.0]], dtype=np.float32)
    inverted = raw.max() + raw.min() - raw
    assert inverted[0, 0] > inverted[-1, -1]
    return {'exact_npy': True, 'aspect_padding': True, 'invalid_rejection': True, 'monochrome1_inversion': True}


def mesh_contract_test():
    grid = np.indices((TARGET_SIZE,) * 3, sparse=True)
    centers = [(82, 128, 142), (150, 128, 118), (124, 164, 146), (154, 92, 118)]
    radii = [18, 17, 10, 8]
    meshes = {}
    for bone, center, radius in zip(BONES, centers, radii):
        distance = sum((axis - coordinate) ** 2 for axis, coordinate in zip(grid, center))
        probability = (distance <= radius ** 2).astype(np.float32)
        meshes[bone] = volume_to_mesh(probability)
        assert meshes[bone] is not None
    scene = build_scene(meshes, ['femur', 'patella'])
    assert set(scene.geometry) == set(BONES)
    assert scene.geometry['femur'].visual.material.baseColorFactor[-1] == BONE_RGBA['femur'][-1]
    assert scene.geometry['tibia'].visual.material.baseColorFactor[-1] == MUTED_RGBA[-1]
    output = UI_TEMP_ROOT / '_mesh_contract.glb'
    try:
        scene.export(file_obj=output, file_type='glb')
        assert output.read_bytes()[:4] == b'glTF'
    finally:
        output.unlink(missing_ok=True)
    empty = np.zeros((TARGET_SIZE,) * 3, dtype=np.float32)
    assert volume_to_mesh(empty) is None
    return {'scene_bones': sorted(scene.geometry), 'glb': True, 'empty_channel': True}


print('factorial contract:', factorial_contract_test())
print('preprocessing contract:', preprocessing_contract_test())
print('mesh contract:', mesh_contract_test())


factorial contract: {'layout': [('plain_unet_style', 'relu', False), ('residual_relu_style', 'relu', True), ('plain_prelu_style', 'prelu', False), ('residual_vnet_style', 'prelu', True)], 'channels': ['femur', 'tibia', 'patella', 'fibula']}
preprocessing contract: {'exact_npy': True, 'aspect_padding': True, 'invalid_rejection': True, 'monochrome1_inversion': True}
mesh contract: {'scene_bones': ['femur', 'fibula', 'patella', 'tibia'], 'glb': True, 'empty_channel': True}


## Local artifact preflight

This local gate hashes candidate artifacts and strictly loads all four decoder states for the first complete fold. The app does not launch if no complete four-arm fold is available.


In [7]:
COMPLETE_FOLDS, FOLD_DIAGNOSTICS, RUN_IDENTITIES = discover_complete_folds()
print(json.dumps({f'fold_{fold}': messages for fold, messages in FOLD_DIAGNOSTICS.items()}, indent=2))
if not COMPLETE_FOLDS:
    raise RuntimeError('No hash-valid complete four-arm fold exists; finish or repair Stage 2.3 before launching the UI.')

DEFAULT_FOLD = COMPLETE_FOLDS[0]
preflight = {}
for arm in DISPLAY_ARMS:
    decoder = load_decoder(RUN_IDENTITIES[DEFAULT_FOLD][arm], device='cpu')
    preflight[arm] = {
        'parameters': sum(parameter.numel() for parameter in decoder.parameters()),
        'checkpoint_sha256': RUN_IDENTITIES[DEFAULT_FOLD][arm]['checkpoint_sha256'][:12],
    }
    del decoder
print({'complete_folds': COMPLETE_FOLDS, 'strict_state_preflight_fold': DEFAULT_FOLD, 'arms': preflight})


{
  "fold_0": [
    "PASS \u2014 complete four-arm fold"
  ],
  "fold_1": [
    "PASS \u2014 complete four-arm fold"
  ],
  "fold_2": [
    "PASS \u2014 complete four-arm fold"
  ],
  "fold_3": [
    "PASS \u2014 complete four-arm fold"
  ],
  "fold_4": [
    "PASS \u2014 complete four-arm fold"
  ]
}
{'complete_folds': [0, 1, 2, 3, 4], 'strict_state_preflight_fold': 0, 'arms': {'plain_unet_style': {'parameters': 8429956, 'checkpoint_sha256': '03f0e05e8324'}, 'residual_relu_style': {'parameters': 8604516, 'checkpoint_sha256': '80ec1af56b67'}, 'plain_prelu_style': {'parameters': 8430916, 'checkpoint_sha256': '1c3c997bdbb1'}, 'residual_vnet_style': {'parameters': 8605476, 'checkpoint_sha256': '27c57f258bcf'}}}


## Four-arm inference callbacks

The frozen front end runs once. Decoders then run sequentially so all four arms see byte-identical features without occupying GPU memory simultaneously.


In [8]:
def arm_probabilities_to_meshes(logits):
    if tuple(logits.shape) != (1, len(BONES), TARGET_SIZE, TARGET_SIZE, TARGET_SIZE):
        raise RuntimeError(f'decoder output contract failed: {tuple(logits.shape)}')
    meshes = {}
    empty = []
    for bone_index, bone in enumerate(BONES):
        probability = torch.sigmoid(logits[0, bone_index].float()).cpu().numpy()
        geometry = volume_to_mesh(probability)
        meshes[bone] = geometry
        if geometry is None:
            empty.append(bone)
        del probability
    return meshes, empty


def run_comparison(fold_label, ap_blob, lat_blob, highlighted, session_id):
    if session_id not in SESSION_CACHE:
        raise RuntimeError('UI session expired; refresh the page and upload the images again.')
    fold = int(str(fold_label).replace('fold_', ''))
    if fold not in COMPLETE_FOLDS:
        raise ValueError(f'fold_{fold} is not a complete verified four-arm fold')

    ap_array, ap_warnings, ap_source = preprocess_upload(ap_blob, 'AP')
    lat_array, lat_warnings, lat_source = preprocess_upload(lat_blob, 'LAT')
    identities = RUN_IDENTITIES[fold]
    shared_hashes = {identity['shared_frontend_sha256'] for identity in identities.values()}
    if len(shared_hashes) != 1:
        raise RuntimeError('selected fold no longer has one shared front-end hash')
    shared_sha = next(iter(shared_hashes))

    ap = torch.from_numpy(ap_array)[None, None].to(DEVICE)
    lat = torch.from_numpy(lat_array)[None, None].to(DEVICE)
    reset_peak_memory_stats()
    overall_start = time.perf_counter()
    front_end = load_frontend(fold, shared_sha)
    with torch.inference_mode(), amp_context():
        front_start = time.perf_counter()
        features = tuple(feature.detach() for feature in front_end(ap, lat))
        front_seconds = time.perf_counter() - front_start
    del ap, lat

    arm_meshes = {}
    arm_seconds = {}
    empty_channels = {}
    for arm in DISPLAY_ARMS:
        decoder = load_decoder(identities[arm], device=DEVICE)
        started = time.perf_counter()
        with torch.inference_mode(), amp_context():
            logits = decoder(features)
        synchronize_device()
        arm_seconds[arm] = time.perf_counter() - started
        arm_meshes[arm], empty_channels[arm] = arm_probabilities_to_meshes(logits)
        del logits, decoder
        clear_device_cache()

    del features
    clear_device_cache()
    peak_bytes = peak_device_memory_bytes()
    total_seconds = time.perf_counter() - overall_start
    SESSION_CACHE[session_id]['meshes'] = arm_meshes
    views = render_scenes(session_id, highlighted)

    warnings = ap_warnings + lat_warnings
    lines = [
        f'### Verified fold_{fold} comparison complete',
        f'- Preprocessing: AP `{ap_source}`; LAT `{lat_source}`',
        f'- Shared front end: `{shared_sha[:12]}`; executed once in {front_seconds:.2f} s',
    ]
    for arm in DISPLAY_ARMS:
        checkpoint = identities[arm]['checkpoint_sha256'][:12]
        empty_note = ', '.join(empty_channels[arm]) if empty_channels[arm] else 'none'
        lines.append(
            f'- **{ARM_FACTORS[arm]["label"]}** (`{arm}`): {arm_seconds[arm]:.2f} s; '
            f'checkpoint `{checkpoint}`; empty channels: {empty_note}'
        )
    lines.extend([
        f'- Total callback time: {total_seconds:.2f} s; device: {device_name}; peak allocated GPU memory: ' + (f'{peak_bytes / 2**30:.2f} GiB' if peak_bytes is not None else 'n/a on CPU'),
        '- Fixed threshold: 0.5. Highlight changes reuse cached meshes and do not rerun inference.',
    ])
    if warnings:
        lines.append('- Warnings: ' + ' '.join(warnings))
    lines.append('\n> Qualitative research output only — no ground-truth scoring and not for diagnosis.')
    return (ap_array, lat_array, *views, '\n'.join(lines))


def update_highlights(highlighted, session_id):
    return render_scenes(session_id, highlighted)


## Launch the local UI

The queue admits one inference callback at a time. `share=False` prevents creation of a public Gradio link. Temporary derived files expire after one hour and are removed when the browser session closes.


In [9]:
def build_ui():
    fold_choices = [f'fold_{fold}' for fold in COMPLETE_FOLDS]
    with gr.Blocks(
        title='Knee reconstruction — full 2×2 decoder comparison',
        delete_cache=(300, 3600),
    ) as demo:
        session_id = gr.State(new_session, time_to_live=3600, delete_callback=cleanup_session)
        gr.Markdown(
            '# Knee reconstruction — full 2×2 decoder comparison\n'
            'One fold, one frozen front end, and four controlled decoder arms. '
            '**Research use only; not for diagnosis.**'
        )
        with gr.Row():
            fold = gr.Dropdown(
                choices=fold_choices,
                value=fold_choices[0],
                label='Verified fold',
            )
            highlighted = gr.CheckboxGroup(
                choices=BONES,
                value=BONES,
                label='Highlighted bones (unselected bones remain as grey context)',
            )
        with gr.Row():
            ap_upload = gr.File(
                label='AP X-ray',
                file_types=['.dcm', '.dicom', '.png', '.jpg', '.jpeg', '.tif', '.tiff', '.npy'],
                type='binary',
            )
            lat_upload = gr.File(
                label='Lateral X-ray',
                file_types=['.dcm', '.dicom', '.png', '.jpg', '.jpeg', '.tif', '.tiff', '.npy'],
                type='binary',
            )
        run_button = gr.Button('Run all four arms', variant='primary')
        with gr.Row():
            ap_preview = gr.Image(label='Preprocessed AP — 256×256', interactive=False, image_mode='L')
            lat_preview = gr.Image(label='Preprocessed lateral — 256×256', interactive=False, image_mode='L')

        gr.Markdown('### ReLU row')
        with gr.Row():
            view_plain_relu = gr.Model3D(
                label='Plain topology — U (plain_unet_style)',
                display_mode='solid',
                clear_color=(0.04, 0.04, 0.05, 1.0),
                height=440,
                interactive=False,
            )
            view_residual_relu = gr.Model3D(
                label='Residual topology (residual_relu_style)',
                display_mode='solid',
                clear_color=(0.04, 0.04, 0.05, 1.0),
                height=440,
                interactive=False,
            )
        gr.Markdown('### PReLU row')
        with gr.Row():
            view_plain_prelu = gr.Model3D(
                label='Plain topology (plain_prelu_style)',
                display_mode='solid',
                clear_color=(0.04, 0.04, 0.05, 1.0),
                height=440,
                interactive=False,
            )
            view_residual_prelu = gr.Model3D(
                label='Residual topology — V (residual_vnet_style)',
                display_mode='solid',
                clear_color=(0.04, 0.04, 0.05, 1.0),
                height=440,
                interactive=False,
            )
        status = gr.Markdown('Upload AP and lateral images, then run all four arms.')
        viewers = [view_plain_relu, view_residual_relu, view_plain_prelu, view_residual_prelu]

        run_button.click(
            fn=run_comparison,
            inputs=[fold, ap_upload, lat_upload, highlighted, session_id],
            outputs=[ap_preview, lat_preview, *viewers, status],
        )
        highlighted.change(
            fn=update_highlights,
            inputs=[highlighted, session_id],
            outputs=viewers,
            queue=False,
        )
    return demo


demo = build_ui()
demo.queue(default_concurrency_limit=1, max_size=4)
demo.launch(
    server_name='127.0.0.1',
    share=False,
    inline=True,
    show_error=True,
    allowed_paths=[str(UI_TEMP_ROOT)],
)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


{'fold': 0, 'front_end_layout': 'local_fold_without_underscore', 'provenance_validation': 'external_json_and_embedded_checkpoint'}


## Local acceptance

Before treating this notebook as executed successfully:

1. Upload one AP/LAT pair and confirm all four rotatable knees appear in the correct factorial positions.
2. Select and clear individual bones; all four viewers must update without another inference run.
3. Confirm the status reports one shared-front-end execution, four arm runtimes, checkpoint hashes, and peak GPU memory.
4. Retain no raw uploads or patient-identifying metadata.
5. Keep raw uploads and identifying DICOM metadata out of saved notebook outputs.
